# ProtT5 Evotuning

we based this of our notbeook published [here](https://github.com/RSchmirler/ProtT5-EvoTuning/tree/main)

## Imports and env. variables

In [1]:
# import dependencies
import os.path

# set path here
os.chdir("path to DeePEn")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss
from torch.utils.data import DataLoader

import re
import numpy as np
import pandas as pd
import copy

import transformers, datasets

from transformers import T5Tokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import DataCollatorForLanguageModeling
from transformers import GenerationConfig
from transformers import TrainingArguments, Trainer, set_seed
from transformers import EarlyStoppingCallback

from typing import Optional, Tuple, Union, Any, List, NewType, Callable, Dict
from collections.abc import Mapping

from transformers.trainer_callback import TrainerCallback

import peft
from peft import get_peft_config, PeftModel, PeftConfig, inject_adapter_in_model, LoraConfig, get_peft_model

from evaluate import load
from datasets import Dataset

from tqdm import tqdm
import random

from Bio import SeqIO
from io import StringIO
import requests

import matplotlib.pyplot as plt

# Environment to run this notebook


These are the versions of the core packages we use to run this notebook:

In [2]:
print("Torch version: ",torch.__version__)
print("Cuda version: ",torch.version.cuda)
print("Numpy version: ",np.__version__)
print("Pandas version: ",pd.__version__)
print("Transformers version: ",transformers.__version__)
print("Datasets version: ",datasets.__version__)

Torch version:  2.7.0+cu126
Cuda version:  12.6
Numpy version:  2.0.2
Pandas version:  2.2.3
Transformers version:  4.51.3
Datasets version:  3.5.1


### Select your model:

In [3]:
checkpoint = "Rostlab/prot_t5_xl_uniref50"

# Input data


In [4]:
def read_data(path):

    # Parse the FASTA file
    records = SeqIO.parse("./data/additional_inputs/Evotuning_msa/" + path +"_msa.fasta", 'fasta')
    # Convert to a DataFrame
    data = [(record.id, str(record.seq)) for record in records]
    df_train = pd.DataFrame(data, columns=['name', 'sequence'])

    
    # Load a Random subset of ~1100 SwissProt sequences
    url = 'https://raw.githubusercontent.com/RSchmirler/ProtT5-EvoTuning/refs/heads/main/data/swissprot_protein_level_subset.tsv'

    response = requests.get(url)
    response.raise_for_status()  # Check if the request was successful

    # Create a StringIO object to simulate a file-like object
    tsv_file = StringIO(response.text)

    # Load the TSV content into a pandas DataFrame
    df_valid = pd.read_csv(tsv_file, sep='\t')


    
    return df_train, df_valid

# Models and Low Rank Adaptation

## T5 Models

### Load T5 model
this creates a T5 model with prediction head and LoRA modification

In [5]:
def load_T5_model(checkpoint):

    # Load model and tokenizer

    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint, force_download=False)
    tokenizer = T5Tokenizer.from_pretrained(checkpoint, force_download=False)

    # Print number of trainable parameters
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    print("T5_EncDec\nTrainable Parameter: "+ str(params))

    # lora modification
    peft_config = LoraConfig(
        r=4, lora_alpha=1, bias="all", target_modules=["q","k","v","o"], task_type = "SEQ_2_SEQ_LM",
    )

    # create peft SEQ_2_SEQ_LM model
    model = get_peft_model(model, peft_config)

    # Print trainable Parameter
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    print("T5_LoRA_EncDec\nTrainable Parameter: "+ str(params) + "\n")

    return model, tokenizer

# Training Definition 

## Training functions

In [6]:
def save_model(model,filepath):
# Saves all parameters that were changed during finetuning

    # Create a dictionary to hold the non-frozen parameters
    non_frozen_params = {}

    # Iterate through all the model parameters
    for param_name, param in model.named_parameters():
        # If the parameter has requires_grad=True, add it to the dictionary
        if param.requires_grad:
            non_frozen_params[param_name] = param

    # Save only the finetuned parameters
    torch.save(non_frozen_params, filepath)


def load_model(checkpoint, filepath):
# Creates a new PT5 model and loads the finetuned weights from a file

    # load model
    model, tokenizer = load_T5_model(checkpoint)

    # Load the non-frozen parameters from the saved file
    non_frozen_params = torch.load(filepath)

    # Assign the non-frozen parameters to the corresponding parameters of the model
    for param_name, param in model.named_parameters():
        if param_name in non_frozen_params:
            param.data = non_frozen_params[param_name].data

    return tokenizer, model

In [7]:
def shift_right(input_ids):
    decoder_start_token_id = 0
    pad_token_id = 0

    shifted_input_ids = torch.full(input_ids.shape[:-1] + (1,), decoder_start_token_id)
    shifted_input_ids = torch.cat([shifted_input_ids, input_ids[..., :-1]], dim=-1)

    return shifted_input_ids


def pad_without_fast_tokenizer_warning(tokenizer, *pad_args, **pad_kwargs):
    """
    Pads without triggering the warning about how using the pad function is sub-optimal when using a fast tokenizer.
    """

    # To avoid errors when using Feature extractors
    if not hasattr(tokenizer, "deprecation_warnings"):
        return tokenizer.pad(*pad_args, **pad_kwargs)

    # Save the state of the warning, then disable it
    warning_state = tokenizer.deprecation_warnings.get("Asking-to-pad-a-fast-tokenizer", False)
    tokenizer.deprecation_warnings["Asking-to-pad-a-fast-tokenizer"] = True

    try:
        padded = tokenizer.pad(*pad_args, **pad_kwargs)
    finally:
        # Restore the state of the warning.
        tokenizer.deprecation_warnings["Asking-to-pad-a-fast-tokenizer"] = warning_state

    return padded

class T5DataCollatorForPretraining(DataCollatorForLanguageModeling):
    def torch_call(self, examples: List[Union[List[int], Any, Dict[str, Any]]]) -> Dict[str, Any]:
        
        # Handle dict or lists with proper padding and conversion to tensor.
        if isinstance(examples[0], Mapping):
            batch = pad_without_fast_tokenizer_warning(
                self.tokenizer, examples, return_tensors="pt", pad_to_multiple_of=self.pad_to_multiple_of
            )

        else:
            batch = {
                "input_ids": _torch_collate_batch(examples, self.tokenizer, pad_to_multiple_of=self.pad_to_multiple_of)
            }

        # If special token mask has been preprocessed, pop it from the dict.
        special_tokens_mask = batch.pop("special_tokens_mask", None)
        if self.mlm:
            batch["input_ids"], batch["labels"], batch["decoder_input_ids"] = self.torch_mask_tokens(
                batch["input_ids"], special_tokens_mask=special_tokens_mask
            )
            
        else:
            labels = batch["input_ids"].clone()
            if self.tokenizer.pad_token_id is not None:
                labels[labels == self.tokenizer.pad_token_id] = -100
            batch["labels"] = labels
        
        return batch    
    
    
    def torch_mask_tokens(self, inputs: Any, special_tokens_mask: Optional[Any] = None) -> Tuple[Any, Any, Any]:
        """
        Prepare masked tokens inputs/labels for masked language modeling: 80% MASK, 10% random, 10% original.
        """
        labels = inputs.clone()
        # We sample a few tokens in each sequence for MLM training (with probability `self.mlm_probability`)
        probability_matrix = torch.full(labels.shape, self.mlm_probability)
        if special_tokens_mask is None:
            special_tokens_mask = [
                self.tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
            ]
            special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        else:
            special_tokens_mask = special_tokens_mask.bool()

        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
        masked_indices = torch.bernoulli(probability_matrix).bool()
        labels[~masked_indices] = -100  # We only compute loss on masked tokens
        
        # Create decoder inputs
        decoder_inputs = shift_right(inputs)
        
        # we replace ALL masked input tokens with tokenizer.mask_token ([MASK])
        inputs[masked_indices] = self.tokenizer.convert_tokens_to_ids(self.tokenizer.mask_token)        

        # 80% of the time, we replace masked input tokens with tokenizer.mask_token ([MASK])
        # indices_replaced = torch.bernoulli(torch.full(labels.shape, 1.0)).bool() & masked_indices
        # inputs[indices_replaced] = self.tokenizer.convert_tokens_to_ids(self.tokenizer.mask_token)

#         # 10% of the time, we replace masked input tokens with random word
#         indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & masked_indices & ~indices_replaced
#         # random_words = torch.randint(len(self.tokenizer), labels.shape, dtype=torch.long)
#         # inputs[indices_random] = random_words[indices_random]
#         custom_token_ids = [i for i in range(3, 28) if i != 23]
        
#         # Convert the custom list to a Torch tensor
#         custom_tensor = torch.tensor(custom_token_ids)
#         # Sample indices using randint over the range of available indices
#         sampled_indices = torch.randint(0, len(custom_token_ids), labels.shape)

#         inputs[indices_random] = custom_tensor[sampled_indices][indices_random]

        # The rest of the time (10% of the time) we keep the masked input tokens unchanged
        return inputs, labels, decoder_inputs

In [8]:
# Set random seeds for reproducibility of your trainings run
def set_seeds(s):
    torch.manual_seed(s)
    np.random.seed(s)
    random.seed(s)
    set_seed(s)

# Dataset creation
def create_dataset(tokenizer,seqs):
    tokenized = tokenizer(seqs, max_length=768, padding=False, truncation=True)
    dataset = Dataset.from_dict(tokenized)

    return dataset


# Main training fuction
def train_per_protein(
        checkpoint,       #model checkpoint
        data_path,        #dataset name
        save_path,        #path to save your model
      
        num_labels = 1,   #1 for regression, >1 for classification
    
        # effective training batch size is batch * accum
        # we recommend an effective batch size of 8 
        batch = 4,        #for training
        accum = 2,        #gradient accumulation
    
        val_batch = 1,   #batch size for evaluation
        epochs = 10,      #training epochs
        lr = 3e-4,        #recommended learning rate
        seed = 42,        #random seed
        mixed = True,     #enable mixed precision training
        gpu = 1 ):        #gpu selection (1 for first gpu)
    

    print("Model used:", checkpoint, "\n")

    # Set gpu device
    os.environ["CUDA_VISIBLE_DEVICES"]=str(gpu-1)
    
    # Set all random seeds
    set_seeds(seed)
    
    # load model

    model, tokenizer = load_T5_model(checkpoint)
    
    #provide the mask token, not the id
    tokenizer.mask_token = "<extra_id_0>"
    
    train_df, valid_df = read_data(data_path)

    # Preprocess inputs
    # Replace uncommon AAs with "X"
    train_df["sequence"]=train_df["sequence"].str.replace('|'.join(["O","B","U","Z","J"]),"X",regex=True)
    valid_df["sequence"]=valid_df["sequence"].str.replace('|'.join(["O","B","U","Z","J"]),"X",regex=True)

    
    # Add spaces between each amino acid for ProtT5 and ProstT5 to correctly use them
    train_df['sequence']=train_df.apply(lambda row : " ".join(row["sequence"]), axis = 1)
    valid_df['sequence']=valid_df.apply(lambda row : " ".join(row["sequence"]), axis = 1)
        
    # Create Datasets (tokenize labels as well)
    train_set=create_dataset(tokenizer,list(train_df['sequence']))
    valid_set=create_dataset(tokenizer,list(valid_df['sequence']))

    val_dict = {"Swiss-Prot": valid_set, "Train": train_set}
    
    # Specify the path where you want to create the folder
    path_to_create = save_path + "/" + data_path + "_" + str(seed)
    
    # Check if the directory already exists
    if not os.path.exists(path_to_create):
        # Create the directory
        os.makedirs(path_to_create)
        print(f"Directory '{path_to_create}' created.")
    else:
        print(f"Directory '{path_to_create}' already exists.")

    # Huggingface Trainer arguments
    args = Seq2SeqTrainingArguments(
        path_to_create,
        eval_strategy = "steps",
        eval_steps = 40,
        logging_strategy = "epoch",
        save_strategy = "no",
        learning_rate = lr,
        lr_scheduler_type = "cosine",
        warmup_steps = 0,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=val_batch,
        gradient_accumulation_steps=accum,
        num_train_epochs=epochs,
        seed = seed,
        fp16 = mixed,
    ) 

    # Metric definition for validation data
    
    def compute_metrics(eval_pred):
        # Unpack logits and labels from EvalPrediction
        logits_tuple, labels = eval_pred
        # Select the first element, assuming it's the logits

        logits = logits_tuple[0]

        # Convert logits and labels to tensors
        logits_tensor = torch.tensor(logits, dtype=torch.float32)
        labels_tensor = torch.tensor(labels, dtype=torch.long)

        # Initialize the mask to select only predictions where labels are not -100
        mask = labels_tensor != -100  

        # Select only the masked elements
        masked_logits = logits_tensor[mask].view(-1, logits_tensor.shape[-1])
        masked_labels = labels_tensor[mask].view(-1)

        # Calculate the loss for masked tokens
        loss_fn = torch.nn.CrossEntropyLoss(reduction='none')
        loss = loss_fn(masked_logits, masked_labels)

        # Compute the average loss
        average_loss = loss.mean().item()

        # Perplexity is the exponential of the average cross-entropy loss
        perplexity = np.exp(average_loss)

        # Calculate accuracy
        # Get the predicted tokens by finding the index of the max logit
        _, predicted_tokens = torch.max(masked_logits, dim=-1)
        # Compare predicted tokens with actual tokens and compute accuracy
        correct_predictions = (predicted_tokens == masked_labels).sum().item()
        accuracy = correct_predictions / masked_labels.size(0)

        return {"perplexity": perplexity, "accuracy": accuracy}
    

    data_collator = T5DataCollatorForPretraining(tokenizer,checkpoint)
    
    #define device 
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    
    #custom callback to save model (after each second epoch)
    class SaveCallback(TrainerCallback):
        def on_epoch_end(self, args, state, control, logs=None, **kwargs):
            
                if int(state.epoch) % 2 == 1:
                    inf_model=kwargs['model'].to(device)
                    inf_model.eval()

                    #save_model
                    save_model(inf_model,path_to_create + "/" + str(state.global_step) + ".pth")
    
    # Trainer          
    trainer = Seq2SeqTrainer(
        model,
        args,
        train_dataset=train_set,
        eval_dataset=val_dict,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[SaveCallback()]
    )

    trainer.train()

    return tokenizer, model, trainer.state.log_history


# Run Training

## Training

In [9]:
os.listdir("./data/raw/")

['CAPSD_AAV2S_Sinai_2021',
 'GFP_AEQVI_Sarkisyan_2016',
 'HIS7_YEAST_Pokusaeva_2019',
 'PHOT_CHLRE_Chen_2023']

In [10]:
data_path = os.listdir("./data/raw/")[3]
data_path

'PHOT_CHLRE_Chen_2023'

In [11]:
save_path = "./models/ProtT5_evotuning/checkpoints"

In [12]:
GPU = 1

In [ ]:
tokenizer, model, history = train_per_protein(checkpoint, data_path, save_path, batch = 1, accum = 8, epochs = 50, seed = 42, mixed = False, gpu=GPU)

# Plots

In [ ]:
loss = [x['loss'] for x in history if 'loss' in x]

# Get spearman (for regression) or accuracy value (for classification)

metric = [x['eval_Swiss-Prot_perplexity'] for x in history if 'eval_Swiss-Prot_perplexity' in x]
metric2 = [x['eval_Train_perplexity'] for x in history if 'eval_Train_perplexity' in x]

acc = [x['eval_Swiss-Prot_accuracy'] for x in history if 'eval_Swiss-Prot_accuracy' in x]
acc2 = [x['eval_Train_accuracy'] for x in history if 'eval_Train_accuracy' in x]

epochs = [x['epoch'] for x in history if 'loss' in x]
epochsv = [x['epoch'] for x in history if 'eval_Swiss-Prot_perplexity' in x]

# Create a figure with two y-axes
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Plot loss and val_loss on the first y-axis
line1 = ax1.plot(epochs, loss, color="orange", label='train_loss')
line2 = ax1.plot(epochsv, metric2, color='coral', label='train_perplexity')
line3 = ax1.plot(epochsv, metric, color='darkred', label='SwissProt_perplexity')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss / Perplexity')

# Plot the computed metric on the second y-axis
line4 = ax2.plot(epochsv, acc, color='royalblue', label='SwissProt_accuracy')
line5 = ax2.plot(epochsv, acc2, color='cyan', label='train_accuracy')
ax2.set_ylabel('Accuracy')
ax2.set_ylim([0, 1])

# Combine the lines from both y-axes and create a single legend
lines = line1 + line2 + line5 + line3 + line4
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='lower left')

# Show the plot
plt.title("Training History - ProtT5 - "+ path)
plt.show()

# Load Evo-tuned model

In [29]:
data_path

'PHOT_CHLRE_Chen_2023'

In [31]:
# load one of the saved model checkpoints
tokenizer, model = load_model(checkpoint, "./models/ProtT5_evotuning/checkpoints/"+ data_path + "_LoRA_evotuned.pth")

T5_EncDec
Trainable Parameter: 2818830336
T5_LoRA_EncDec
Trainable Parameter: 5900288



In [32]:
# Optional:
# merge the trained LoRA adapter into the original model
# this can be useful to continue working with the Evo-Tuned model (e.g. for fine-tuning it on some downstream task)
model = model.cpu().merge_and_unload('default')

Unloading and merging model: 100%|██████████| 1357/1357 [00:01<00:00, 734.47it/s]
